# Generate Stories for Human Evaluation

Run in Google Colab with GPU (T4 or better).

Outputs `stories.py` — a static file with 5 stories and 3 responses each (H=Human, B=Base LLM, F=Fine-tuned LLM).
Copy the output `stories.py` into `human_eval/stories.py` before deploying the Streamlit app.

In [ ]:
# Install dependencies (Colab)
!pip install -q transformers peft==0.18.0 accelerate bitsandbytes pandas openpyxl
# peft 0.18.0 required — LoRA adapter was saved with this version

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import torch
import random
import textwrap
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# ── CONFIG ──────────────────────────────────────────────────────────────────
MODEL_ID  = "meta-llama/Llama-3.2-1B-Instruct"
LORA_DIR  = "/content/drive/My Drive/Applied_Project/gen_response_model/llama32_1b_empathy_lora"
DATA_PATH = "/content/drive/My Drive/Applied_Project/combined_data_set.csv"  # adjust if needed
HF_TOKEN  = ""  # paste your HuggingFace token here if model is gated

SYSTEM_PROMPT = (
    "You are an empathetic assistant. "
    "Respond with warmth, validation, and emotional support. "
    "Keep it concise and human."
)
MAX_RESPONSE_WORDS = 100  # trim to avoid length bias between conditions
N_STORIES = 5

In [ ]:
# ── LOAD TOKENIZER ───────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# ── LOAD BASE MODEL ──────────────────────────────────────────────────────────
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    token=HF_TOKEN,
)
base_model.eval()
print("Base model loaded")

In [ ]:
# ── LOAD LORA MODEL ──────────────────────────────────────────────────────────
base_for_lora = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    token=HF_TOKEN,
)
lora_model = PeftModel.from_pretrained(base_for_lora, LORA_DIR, low_cpu_mem_usage=False)
lora_model.eval()
print("LoRA model loaded")

In [ ]:
def generate(model, story, max_new_tokens=160):
    """Generate a response. do_sample=False for deterministic output."""
    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Story:\n{story}\n\nWrite an empathetic response."},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return decoded.strip()


def trim_to_words(text, max_words=MAX_RESPONSE_WORDS):
    """Trim response to max_words to reduce length bias between conditions."""
    words = text.split()
    if len(words) <= max_words:
        return text
    return " ".join(words[:max_words]) + "..."

In [ ]:
# ── SELECT 5 STORIES ─────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
print(f"Dataset: {len(df)} rows, columns: {df.columns.tolist()}")

# Filter: original human responses, moderate story length
orig = df[df["source"] == "original"].copy()
orig["story_word_count"] = orig["Story"].str.split().str.len()
candidates = orig[(orig["story_word_count"] >= 50) & (orig["story_word_count"] <= 120)]
print(f"Candidates after length filter: {len(candidates)}")

# Sample 5 with fixed seed for reproducibility
random.seed(42)
selected = candidates.sample(n=N_STORIES, random_state=42).reset_index(drop=True)
print(f"\nSelected {N_STORIES} stories:")
for i, row in selected.iterrows():
    print(f"  [{i+1}] {row['story_word_count']} words — {row['Story'][:80]}...")

In [ ]:
# ── GENERATE RESPONSES ───────────────────────────────────────────────────────
results = []

for i, row in selected.iterrows():
    story = row["Story"].strip()
    human_response = trim_to_words(row["Response"].strip())

    print(f"\n[{i+1}/{N_STORIES}] Generating base response...")
    base_response = trim_to_words(generate(base_model, story))

    print(f"[{i+1}/{N_STORIES}] Generating LoRA response...")
    lora_response = trim_to_words(generate(lora_model, story))

    results.append({
        "id": i + 1,
        "story": story,
        "responses": {
            "H": human_response,
            "B": base_response,
            "F": lora_response,
        }
    })
    print(f"  H: {human_response[:80]}...")
    print(f"  B: {base_response[:80]}...")
    print(f"  F: {lora_response[:80]}...")

print("\nDone.")

In [ ]:
# ── WRITE stories.py ─────────────────────────────────────────────────────────
import json

lines = ["# Auto-generated by generate_stories.ipynb — do not edit manually\n"]
lines.append("STORIES = [\n")
for r in results:
    lines.append(f"    {{\n")
    lines.append(f'        "id": {r["id"]},\n')
    lines.append(f'        "story": {json.dumps(r["story"], ensure_ascii=False)},\n')
    lines.append(f'        "responses": {{\n')
    lines.append(f'            "H": {json.dumps(r["responses"]["H"], ensure_ascii=False)},\n')
    lines.append(f'            "B": {json.dumps(r["responses"]["B"], ensure_ascii=False)},\n')
    lines.append(f'            "F": {json.dumps(r["responses"]["F"], ensure_ascii=False)},\n')
    lines.append(f'        }},\n')
    lines.append(f'    }},\n')
lines.append("]\n")

out_path = "/content/stories.py"
with open(out_path, "w", encoding="utf-8") as f:
    f.writelines(lines)

print(f"Written to {out_path}")
print("Copy this file to human_eval/stories.py before deploying the Streamlit app.")

In [ ]:
# ── OPTIONAL: download from Colab ────────────────────────────────────────────
from google.colab import files
files.download("/content/stories.py")

In [ ]:
# Save stories.py to Google Drive so it's accessible locally
import shutil
drive_out = "/content/drive/My Drive/Applied_Project/stories.py"
shutil.copy("/content/stories.py", drive_out)
print(f"Saved to Drive: {drive_out}")